# L02 · Scenes, Entities, and the Simulation Lifecycle

> **Course status:** the lab has passed clean-kernel verification on CPU and on the reference AMD ROCm platform, including offscreen rendering. L02 is `cpu-verified`.

This notebook turns the lecture's declaration → build → runtime model into observable evidence using one Plane and one falling Box. Run every cell in order from a clean kernel.


## Before you run

The simulation backend and rendering switch are independent:

- `ROBO_GENESIS_BACKEND=auto` is the default. It selects the verified AMD ROCm backend when available and otherwise uses CPU.
- `ROBO_GENESIS_BACKEND=cpu` forces the minimum CPU path.
- `ROBO_GENESIS_RENDER=0` disables camera rendering and produces a state-derived schematic.
- `ROBO_GENESIS_RENDER=1` enables the offscreen camera. A rendering error then fails the run; it does not silently fall back.

Changing either setting requires a fresh kernel because `gs.init()` is process-wide initialization.

Before running the first code cell, write down three predictions:

1. Will `box.get_pos()` work as soon as `scene.add_entity(...)` returns a Box handle, or only after `scene.build()`?
2. After 20 steps, which evidence should change: position, quaternion, linear velocity, or the scene topology?
3. If camera rendering is disabled, what numerical evidence would still convince you that the Box moved?

Keep these predictions nearby. The comparison figure and reflection prompts near the end will help you check them.


In [ ]:
import os

import genesis as gs
import matplotlib.pyplot as plt
import numpy as np
import torch

from robo_genesis.course_manifest import load_course_manifest
from robo_genesis.course_utils import (
    environment_report,
    notebook_mode,
    select_backend,
    to_numpy,
)

lesson = load_course_manifest().lesson("L02")
print(f"lesson_status: {lesson.status.value}")

backend_mode = os.environ.get("ROBO_GENESIS_BACKEND", "auto").strip().lower()
if backend_mode not in {"auto", "cpu"}:
    raise ValueError("ROBO_GENESIS_BACKEND must be 'auto' or 'cpu'")

render_value = os.environ.get("ROBO_GENESIS_RENDER", "0").strip()
if render_value not in {"0", "1"}:
    raise ValueError("ROBO_GENESIS_RENDER must be '0' or '1'")
render_enabled = render_value == "1"

runtime = notebook_mode("l02-one-box-lifecycle", show_viewer=False)
output_dir = runtime["output_dir"]
environment = environment_report()
selected_backend = select_backend(prefer_rocm=backend_mode == "auto")

if backend_mode == "cpu":
    selection_reason = "forced CPU mode requested by ROBO_GENESIS_BACKEND"
elif getattr(gs, "amdgpu", None) is not None and selected_backend == gs.amdgpu:
    selection_reason = "auto mode found a verified ROCm device"
else:
    selection_reason = "auto mode found no verified ROCm path; using CPU"

print(f"backend_mode: {backend_mode}")
print(f"selected_backend: {selected_backend}")
print(f"selection_reason: {selection_reason}")
print(f"render_enabled: {render_enabled}")
print(f"output_dir: {output_dir.resolve()}")

gs.init(backend=selected_backend, logging_level="warning")
assert gs.backend == selected_backend
actual_backend = f"{gs.backend}"
print(f"actual_backend: {actual_backend}")


## 1. Declare the scene

The Plane and Box are built-in primitives. The Box declaration keeps Morph, Material, and Surface visible, while the camera is added only when rendering was enabled before the topology is finalized.


In [ ]:
BOX_SIZE = (0.10, 0.10, 0.10)
BOX_INITIAL_POS = (0.0, 0.0, 0.50)
BOX_DENSITY = 500.0
BOX_COLOR = (0.20, 0.60, 0.90, 1.0)
CAMERA_RESOLUTION = (640, 360)

scene = gs.Scene(
    show_viewer=runtime["show_viewer"],
    sim_options=gs.options.SimOptions(dt=0.01, substeps=2),
)
ground = scene.add_entity(
    morph=gs.morphs.Plane(),
    name="ground",
)
box = scene.add_entity(
    morph=gs.morphs.Box(size=BOX_SIZE, pos=BOX_INITIAL_POS),
    material=gs.materials.Rigid(rho=BOX_DENSITY),
    surface=gs.surfaces.Default(color=BOX_COLOR),
    name="falling_box",
)

camera = None
if render_enabled:
    camera = scene.add_camera(
        res=CAMERA_RESOLUTION,
        pos=(1.1, -1.1, 0.8),
        lookat=(0.0, 0.0, 0.25),
        fov=40,
        GUI=False,
    )

assert scene.is_built is False
assert (camera is not None) == render_enabled
print(f"scene.is_built: {scene.is_built}")
print(f"entity handle: {type(box).__name__}")
print(f"Morph: Box(size={BOX_SIZE}, pos={BOX_INITIAL_POS})")
print(f"Material: Rigid(rho={BOX_DENSITY})")
print(f"Surface: Default(color={BOX_COLOR})")
print(f"camera declared: {camera is not None}")


## 2. Test the pre-build boundary

The Box handle exists, but dynamic state must still be unavailable. Only the expected Genesis exception and message count as a pass.


In [ ]:
pre_build_guard_passed = False
try:
    box.get_pos()
except gs.GenesisException as exc:
    if "not built yet" not in str(exc).lower():
        raise
    pre_build_guard_passed = True
    print("PASS — state is unavailable before build")
else:
    raise AssertionError("get_pos unexpectedly succeeded before scene.build()")


## 3. Build once and inspect the hierarchy

The build call crosses the lifecycle boundary. After it succeeds, inspect this primitive rather than assuming every rigid entity has the same number of links or collision geoms.


In [ ]:
scene.build()
assert scene.is_built is True
print("PASS — scene.is_built changed from False to True")


In [ ]:
entity_type = type(box).__name__
link_count = box.n_links
link_list_count = len(box.links)
geom_count = box.n_geoms
geom_list_count = len(box.geoms)

assert entity_type == "RigidEntity"
assert link_count == link_list_count == 1
assert geom_count == geom_list_count == 1
print(
    "PASS — this Box primitive has "
    f"{link_count} RigidLink and {geom_count} collision RigidGeom"
)
print("This count describes this primitive, not every Genesis Entity.")


## 4. Step and inspect state

Record position, quaternion, and linear velocity before and after exactly 20 outer steps. When rendering is enabled, also capture the initial camera frame before stepping. The checks use shape, device, finiteness, and a directional prediction instead of a fabricated exact final coordinate.


In [ ]:
def checked_state(name, value, expected_shape):
    if not isinstance(value, torch.Tensor):
        raise TypeError(f"{name} must be a torch.Tensor, got {type(value).__name__}")
    actual_shape = tuple(value.shape)
    if actual_shape != expected_shape:
        raise AssertionError(f"{name} shape {actual_shape} != {expected_shape}")
    if not bool(torch.isfinite(value).all().item()):
        raise AssertionError(f"{name} contains non-finite values: {value}")
    print(f"PASS — {name}: shape={actual_shape}, device={value.device}, finite=True")
    return value.detach().clone()


def checked_rgb(name, value):
    rgb = to_numpy(value)
    if rgb.ndim != 3:
        raise AssertionError(f"{name} must be 3D, got shape {rgb.shape}")
    expected_width, expected_height = CAMERA_RESOLUTION
    if rgb.shape[:2] != (expected_height, expected_width):
        raise AssertionError(
            f"{name} height/width {rgb.shape[:2]} != {(expected_height, expected_width)}"
        )
    if rgb.shape[2] not in (3, 4):
        raise AssertionError(f"{name} channel count must be 3 or 4, got {rgb.shape[2]}")
    if not np.isfinite(rgb).all():
        raise AssertionError(f"{name} contains non-finite values")

    rgb_min = float(rgb.min())
    rgb_max = float(rgb.max())
    if np.issubdtype(rgb.dtype, np.integer):
        dtype_max = float(np.iinfo(rgb.dtype).max)
        if not 0.0 <= rgb_min <= rgb_max <= dtype_max:
            raise AssertionError(f"{name} range [{rgb_min}, {rgb_max}] is invalid for {rgb.dtype}")
    elif np.issubdtype(rgb.dtype, np.floating):
        if not 0.0 <= rgb_min <= rgb_max <= 1.0 + 1e-6:
            raise AssertionError(f"{name} range [{rgb_min}, {rgb_max}] is invalid for {rgb.dtype}")
    else:
        raise TypeError(f"unsupported {name} dtype: {rgb.dtype}")
    print(f"PASS — {name}: shape={rgb.shape}, dtype={rgb.dtype}, range=[{rgb_min}, {rgb_max}]")
    return rgb


initial_pos = checked_state("initial_pos", box.get_pos(), (3,))
initial_quat = checked_state("initial_quat", box.get_quat(), (4,))
initial_vel = checked_state("initial_vel", box.get_vel(), (3,))
initial_rgb = None
if render_enabled:
    assert camera is not None
    initial_rgb = checked_rgb("initial_rgb", camera.render(rgb=True)[0])
trajectory = [initial_pos]

for _ in range(20):
    scene.step()
    trajectory.append(box.get_pos().detach().clone())

final_pos = checked_state("final_pos", box.get_pos(), (3,))
final_quat = checked_state("final_quat", box.get_quat(), (4,))
final_vel = checked_state("final_vel", box.get_vel(), (3,))
initial_z = float(initial_pos[2].detach().cpu())
final_z = float(final_pos[2].detach().cpu())
if not final_z < initial_z:
    raise AssertionError(f"expected final_z < initial_z, got {final_z} >= {initial_z}")
height_drop = initial_z - final_z
print(f"PASS — Box height decreased by {height_drop:.6f} m over 20 steps")


## 5. Observe without overstating the evidence

When rendering is enabled, validate and compare camera frames from before and after stepping. When rendering is disabled, report an explicit skip and compare the measured initial/final Box states beside the height trajectory. The two branches make different claims.


In [ ]:
if render_enabled:
    assert camera is not None
    assert initial_rgb is not None
    final_rgb = checked_rgb("final_rgb", camera.render(rgb=True)[0])
    rgb_min = min(float(initial_rgb.min()), float(final_rgb.min()))
    rgb_max = max(float(initial_rgb.max()), float(final_rgb.max()))

    observation_path = output_dir / "l02-camera-comparison.png"
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for axis, frame, title in zip(
        axes,
        (initial_rgb, final_rgb),
        (f"Before stepping: z={initial_z:.3f} m", f"After 20 steps: z={final_z:.3f} m"),
        strict=True,
    ):
        axis.imshow(frame)
        axis.set_title(title)
        axis.axis("off")
    fig.suptitle("Genesis offscreen RGB: compare prediction with observation")
    fig.savefig(observation_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    render_result = (
        f"PASS: initial/final RGB shape={final_rgb.shape}, dtype={final_rgb.dtype}, "
        f"combined range=[{rgb_min}, {rgb_max}]"
    )
    print(render_result)
else:
    print("SKIP: rendering disabled for this run")
    positions = np.stack([to_numpy(position) for position in trajectory])
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

    snapshot_axis = axes[0]
    snapshot_axis.axhline(0.0, color="black", linewidth=1.5, label="Plane z=0")
    box_width, box_height = BOX_SIZE[0], BOX_SIZE[2]
    for label, position, color, alpha in (
        ("initial", to_numpy(initial_pos), "#2A9D8F", 0.35),
        ("final", to_numpy(final_pos), "#E76F51", 0.75),
    ):
        snapshot_axis.add_patch(
            plt.Rectangle(
                (position[0] - box_width / 2, position[2] - box_height / 2),
                box_width,
                box_height,
                color=color,
                alpha=alpha,
                label=f"{label}: z={position[2]:.3f} m",
            )
        )
    snapshot_axis.annotate(
        "observed drop",
        xy=(0.08, final_z),
        xytext=(0.08, initial_z),
        arrowprops={"arrowstyle": "->", "color": "#555555"},
        ha="center",
    )
    snapshot_axis.set(
        xlim=(-0.18, 0.18),
        ylim=(-0.02, max(0.65, initial_z + 0.1)),
        xlabel="x position (m)",
        ylabel="z position (m)",
        title="Initial and final state",
    )
    snapshot_axis.legend(loc="upper left")
    snapshot_axis.grid(alpha=0.2)

    trajectory_axis = axes[1]
    step_indices = np.arange(len(positions))
    trajectory_axis.plot(
        step_indices,
        positions[:, 2],
        color="#457B9D",
        marker="o",
        markersize=3,
    )
    trajectory_axis.scatter(
        (0, 20),
        (initial_z, final_z),
        color=("#2A9D8F", "#E76F51"),
        zorder=3,
    )
    trajectory_axis.annotate(
        f"initial {initial_z:.3f} m",
        (0, initial_z),
        xytext=(12, -20),
        textcoords="offset points",
    )
    trajectory_axis.annotate(
        f"final {final_z:.3f} m",
        (20, final_z),
        xytext=(-82, 12),
        textcoords="offset points",
    )
    trajectory_axis.set(
        xlabel="outer step",
        ylabel="Box center z (m)",
        title="Measured height over 20 steps",
    )
    trajectory_axis.grid(alpha=0.3)
    fig.suptitle("Measured Genesis state (not a camera render)")
    observation_path = output_dir / "l02-state-comparison.png"
    fig.savefig(observation_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    render_result = "SKIP: rendering disabled; state-derived comparison and trajectory only"
    print(f"state comparison: {observation_path.resolve()}")


### Compare the evidence with your prediction

- Did the initial and final panels match your predicted direction of motion? Point to the numerical value that supports your answer.
- Which observations came from Genesis state tensors, and which came from the optional camera? What can each source prove?
- Compare position, quaternion, and linear velocity. Which values changed, and why is an unchanged value still useful evidence?


## 6. Test the closed topology

After build, adding a new entity must fail with the expected lifecycle error. As before, an unrelated exception is not accepted as evidence.


In [ ]:
post_build_guard_passed = False
try:
    scene.add_entity(
        morph=gs.morphs.Sphere(radius=0.05),
        name="too_late",
    )
except gs.GenesisException as exc:
    if "already built" not in str(exc).lower():
        raise
    post_build_guard_passed = True
    print("PASS — topology declaration is closed after build")
else:
    raise AssertionError("add_entity unexpectedly succeeded after scene.build()")


## Evidence summary

The final summary separates the core simulation result from the optional rendering result. A rendering skip is not reported as a rendering pass.


In [ ]:
assert pre_build_guard_passed
assert post_build_guard_passed
assert scene.is_built is True
assert link_count == geom_count == 1
assert final_z < initial_z

evidence = {
    "genesis_world": environment["genesis_world"],
    "backend_mode": backend_mode,
    "actual_backend": actual_backend,
    "render_enabled": render_enabled,
    "render_result": render_result,
    "scene_built": scene.is_built,
    "entity_type": entity_type,
    "links": link_count,
    "collision_geoms": geom_count,
    "initial_z": initial_z,
    "final_z": final_z,
    "height_drop": height_drop,
    "observation_artifact": str(observation_path),
}
for key, value in evidence.items():
    print(f"{key:>22}: {value}")
print("PASS — core L02 lifecycle evidence is complete")


## Exercise

Add this fixed marker in the declaration cell, before `scene.build()`:

```python
marker = scene.add_entity(
    morph=gs.morphs.Box(
        size=(0.06, 0.06, 0.06),
        pos=(0.18, 0.0, 0.03),
        fixed=True,
    ),
    surface=gs.surfaces.Default(color=(0.20, 0.80, 0.35, 1.0)),
    name="fixed_marker",
)
```

Restart the kernel and run the notebook from the top. Inspect the marker's entity and link structure; when rendering is enabled, also confirm it appears in the image. Explain which arguments belong to Morph and which belong to Surface.


## Connection to L03

This experiment kept the physics configuration fixed so that lifecycle was the only variable under study. L03 will vary timestep, substeps, contact, and friction and measure how those choices affect rigid-body behavior.
